In [ ]:
import torch
import torch.nn as nn
import torchvision
from torch.utils.data import DataLoader
import torchvision.transforms.v2 as v2
import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm


class VAE(nn.Module):
    def __init__(self, latent_dim=64):
        super(VAE, self).__init__()

        self.encoder = nn.Sequential(
            nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, stride=1, padding=1),  # 3*32*32 -> 16*32*32
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),  # 16*32*32 -> 16*16*16

            nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, stride=1, padding=1),  # 16*16*16 -> 32*16*16
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),  # 32*16*16 -> 32*8*8

            nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, stride=1, padding=1),  # 32*8*8 -> 64*8*8
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),  # 64*8*8 -> 64*4*4

            nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, stride=1, padding=1),  # 64*4*4 -> 128*4*4
            nn.ReLU(),
        )

        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(in_channels=128, out_channels=64, kernel_size=2, stride=2),  # 128*4*4 -> 64*8*8
            nn.ReLU(),

            nn.ConvTranspose2d(in_channels=64, out_channels=32, kernel_size=2, stride=2),  # 64*8*8 -> 32*16*16
            nn.ReLU(),

            nn.ConvTranspose2d(in_channels=32, out_channels=3, kernel_size=2, stride=2),  # 32*16*16 -> 3*32*32
            nn.Sigmoid(),
        )

        #################################
        ###### VAE Special layers #######
        #################################

        # Encoder
        self.flatten = nn.Flatten()
        self.fc_mu = nn.Linear(in_features=128 * 4 * 4, out_features=latent_dim)
        self.fc_logvar = nn.Linear(in_features=128 * 4 * 4, out_features=latent_dim)

        # Decoder
        self.decoder_input = nn.Linear(in_features=latent_dim, out_features=128 * 4 * 4)

    # expectation over the distribution
    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def forward(self, x):
        x = self.encoder(x)

        x = self.flatten(x)
        mu = self.fc_mu(x)
        logvar = self.fc_logvar(x)

        z = self.reparameterize(mu, logvar)
        x = self.decoder_input(z)
        x = x.view(x.shape[0], 128, 4, 4)

        x = self.decoder(x)

        return x, mu, logvar


transform = v2.ToTensor()
cifar10_train_dataset = torchvision.datasets.CIFAR10(root='./', download=True, train=True, transform=transform)
cifar10_test_dataset = torchvision.datasets.CIFAR10(root='./', download=True, train=False, transform=transform)

train_loader = DataLoader(cifar10_train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(cifar10_test_dataset, batch_size=64, shuffle=False)


def loss_function(recon_x, x, mu, logvar, alpha=0.1):
    # loss reconstruction
    loss_recon = torch.nn.MSELoss()(recon_x, x)
    # loss KL Divergence
    # Our intuition of Gaussian distribution VS N(0,1) in 64 dimension
    loss_kl = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())

    total_loss = (1 - alpha) * loss_recon + alpha * loss_kl
    return total_loss, loss_recon, loss_kl


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = VAE(latent_dim=64)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
criterion = loss_function

model = model.to(device)

for epoch in range(10):
    losses = {
        "total": [],
        "recon": [],
        "kl": []
    }
    for imgs, labels in tqdm(train_loader):
        imgs = imgs.to(device)
        optimizer.zero_grad()

        outs, mu, logvar = model(imgs)

        loss, loss_recon, loss_kl = criterion(outs, imgs, mu, logvar)
        loss.backward()
        optimizer.step()
        losses['total'].append(loss.item())
        losses['recon'].append(loss_recon.item())
        losses['kl'].append(loss_kl.item())

    print("epoch: ", epoch, "loss: ", np.mean(losses['total']), "recon: ", np.mean(losses['recon']), "kl: ",
          np.mean(losses['kl']))
for imgs, _ in test_loader:
    break

outs, _, _ = model(imgs.to(device))
idx = 3
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 5))

ax1.imshow(imgs[idx].permute(1, 2, 0))
ax1.set_title("Original")

ax2.imshow(outs[idx].permute(1, 2, 0).cpu().detach())
ax2.set_title("Reconstructed!")
